# 3. Validate Global N-Gram DuckDB Store

This notebook validates and inspects the global DuckDB database built by notebook 2.

Notation:

- `V_n`: exact chord n-gram vocabulary, keyed by `exact_ngram_id`.
- `H_n`: harmonic vocabulary, keyed by `harmonic_id`.
- `harmonic_key`: serialized first-root-normalized `12 x n` matrix, retained for reconstruction/debugging.

Notebook 2 owns database creation. This notebook checks mass preservation, vocabulary collapse, and example queries before downstream analysis.

## Pipeline Role

This notebook is the quality-control layer for the global n-gram store.

It preserves the distinction between global objects and grouped document counts:

- global `V_n` and `H_n` live in `exact_ngrams` and `harmonic_ngrams`;
- target-limited stratified trends are built in notebook 4;
- broad document-term counts are built in notebook 7.

## Design

Notebook 2 streams the raw songs and writes `data/processed/harmonic_trends.duckdb` directly. This notebook does not rebuild or import CSV files.

It checks:

1. Required core tables are present.
2. Total occurrence mass is preserved from `V_n` to `H_n`.
3. Harmonic collapse reduces unique row counts while preserving total counts.
4. Common harmonic classes and their exact representatives look plausible.

Important: `V_count == H_count` is expected. The harmonic collapse reduces the number of unique classes, not the total number of n-gram occurrences.

In [1]:
from pathlib import Path
import importlib
import sys

import duckdb

CWD = Path.cwd()
ROOT = CWD.parent if (CWD / "utils").exists() else CWD
NOTEBOOK_DIR = ROOT / "notebooks"

# Force this repo's notebook utilities to win over any stale or external `utils` package.
sys.path = [p for p in sys.path if p != str(NOTEBOOK_DIR)]
sys.path.insert(0, str(NOTEBOOK_DIR))
for module_name in list(sys.modules):
    if module_name == "utils" or module_name.startswith("utils."):
        del sys.modules[module_name]

from utils import duckdb_store as ds
ds = importlib.reload(ds)

expected_duckdb_store = (NOTEBOOK_DIR / "utils" / "duckdb_store.py").resolve()
loaded_duckdb_store = Path(ds.__file__).resolve()
assert loaded_duckdb_store == expected_duckdb_store, (
    f"Imported wrong duckdb_store module: {loaded_duckdb_store}. "
    f"Expected: {expected_duckdb_store}. Restart the kernel and rerun from the top."
)

{
    "python": sys.executable,
    "duckdb_version": duckdb.__version__,
    "duckdb_store": str(loaded_duckdb_store),
}

{'python': '/usr/local/bin/python3',
 'duckdb_version': '1.5.2',
 'duckdb_store': '/Users/juansalinas/Documents/GitHub/harmonic-trends/notebooks/utils/duckdb_store.py'}

In [2]:
DB_PATH = ROOT / "data" / "processed" / "harmonic_trends.duckdb"

assert DB_PATH.exists(), f"Missing notebook 2 database: {DB_PATH}"
ROOT, DB_PATH

(PosixPath('/Users/juansalinas/Documents/GitHub/harmonic-trends'),
 PosixPath('/Users/juansalinas/Documents/GitHub/harmonic-trends/data/processed/harmonic_trends.duckdb'))

## Preflight Notebook 2 Database

In [3]:
required_tables = {
    "exact_ngrams",
    "harmonic_ngrams",
    "exact_to_harmonic",
    "frequency_validation",
    "song_ngram_summary",
    "ngram_run_summary",
    "harmonic_parse_errors",
    "metadata",
    "ngram_distances",
}

con = duckdb.connect(str(DB_PATH), read_only=True)
ds.configure_connection(con)
try:
    tables = set(con.execute("SHOW TABLES").fetchdf()["name"])
    missing = sorted(required_tables - tables)
    assert not missing, f"Run notebook 2 first. Missing tables/views: {missing}"

    exact_preview = con.execute(
        """
        SELECT *
        FROM exact_ngrams
        ORDER BY n, count DESC, exact_ngram_id
        LIMIT 5
        """
    ).fetchdf()
    exact_columns = set(exact_preview.columns)
finally:
    con.close()

required_columns = {
    "exact_ngram_id",
    "n",
    "ngram_json",
    "ngram",
    "count",
    "frequency",
    "harmonic_key",
    "harmonic_id",
}
missing_columns = sorted(required_columns - exact_columns)
assert not missing_columns, f"exact_ngrams is missing columns: {missing_columns}"

exact_preview

,exact_ngram_id,n,ngram_json,ngram,count,frequency,harmonic_key,harmonic_id
0,3_ce87d78ae19f8322,3,"[""G"", ""C"", ""G""]",G C G,522905,0.010327,3:111000000000101010000101000010000000,H3_ede3c4f53675bbb0
1,3_6adafd20068442f4,3,"[""C"", ""G"", ""D""]",C G D,459111,0.009067,3:100000011000100000001110000001000010,H3_4e99eecf61b0edec
2,3_8236526a0d85f6fd,3,"[""C"", ""G"", ""C""]",C G C,451349,0.008914,3:101000010000101000000111000000000010,H3_35d6bfda85b78e3b
3,3_b2f9fe8f46eb2ef6,3,"[""F"", ""C"", ""G""]",F C G,403830,0.007975,3:100000011000100000001110000001000010,H3_4e99eecf61b0edec
4,3_fe4dbe8dc705a25b,3,"[""D"", ""G"", ""D""]",D G D,396222,0.007825,3:111000000000101010000101000010000000,H3_ede3c4f53675bbb0


## Validate Mass Preservation

In [4]:
validation = ds.validate_harmonic_trends_database(DB_PATH)
validation

,check_name,n,passed,exact_count,harmonic_count
0,exact_count_equals_harmonic_count,3.0,True,50635145.0,50635145.0
1,exact_count_equals_harmonic_count,4.0,True,49955522.0,49955522.0
2,exact_count_equals_harmonic_count,5.0,True,49276039.0,49276039.0
3,exact_count_equals_harmonic_count,6.0,True,48596846.0,48596846.0
4,exact_count_equals_harmonic_count,7.0,True,47918358.0,47918358.0
5,exact_count_equals_harmonic_count,8.0,True,47240720.0,47240720.0
6,harmonic_id_collision_free,NaN,True,NaN,NaN
7,exact_to_harmonic_references_exist,NaN,True,0.0,NaN


The pushforward from `V_n` to `H_n` should preserve total occurrence mass. Counts should match because every exact occurrence maps to exactly one harmonic class.

In [5]:
assert validation["passed"].all(), validation

In [6]:
assert validation["passed"].all()
"database validation passed"

'database validation passed'

## Inspect Size and Collapse

Use this section to distinguish total occurrence mass from unique vocabulary size. The total counts in `V_n` and `H_n` should match; the unique row counts should shrink after harmonic collapse.

`representative_example_ngram` is a display example for the harmonic class, not a new canonical musical name.

In [7]:
con = duckdb.connect(str(DB_PATH), read_only=True)
try:
    ds.configure_connection(con)
    row_counts = con.execute(
        """
        SELECT 'exact_ngrams' AS table_name, COUNT(*) AS row_count FROM exact_ngrams
        UNION ALL SELECT 'harmonic_ngrams', COUNT(*) FROM harmonic_ngrams
        UNION ALL SELECT 'exact_to_harmonic', COUNT(*) FROM exact_to_harmonic
        UNION ALL SELECT 'song_ngram_summary', COUNT(*) FROM song_ngram_summary
        UNION ALL SELECT 'ngram_run_summary', COUNT(*) FROM ngram_run_summary
        UNION ALL SELECT 'frequency_validation', COUNT(*) FROM frequency_validation
        UNION ALL SELECT 'ngram_distances', COUNT(*) FROM ngram_distances
        ORDER BY table_name
        """
    ).fetchdf()

    collapse_summary = con.execute(
        """
        SELECT
            v.n,
            v.unique_V,
            h.unique_H,
            v.unique_V - h.unique_H AS collapsed_unique_types,
            h.unique_H::DOUBLE / NULLIF(v.unique_V, 0) AS H_over_V,
            h.avg_exact_types_per_H,
            h.max_exact_types_per_H,
            v.total_occurrences AS V_count,
            h.total_occurrences AS H_count,
            v.total_occurrences = h.total_occurrences AS counts_match
        FROM (
            SELECT n, COUNT(*) AS unique_V, SUM(count)::BIGINT AS total_occurrences
            FROM exact_ngrams
            GROUP BY n
        ) v
        JOIN (
            SELECT
                n,
                COUNT(*) AS unique_H,
                SUM(count)::BIGINT AS total_occurrences,
                AVG(n_exact_ngrams) AS avg_exact_types_per_H,
                MAX(n_exact_ngrams) AS max_exact_types_per_H
            FROM harmonic_ngrams
            GROUP BY n
        ) h USING (n)
        ORDER BY n
        """
    ).fetchdf()
finally:
    con.close()

row_counts, collapse_summary

(             table_name  row_count
 0          exact_ngrams   34464863
 1     exact_to_harmonic   34464863
 2  frequency_validation          6
 3       harmonic_ngrams   20696047
 4       ngram_distances          0
 5     ngram_run_summary          6
 6    song_ngram_summary     679807,
    n  unique_V  unique_H  collapsed_unique_types  H_over_V  \
 0  3    828417    192951                  635466  0.232915   
 1  4   2250695    843307                 1407388  0.374687   
 2  5   4269570   2081609                 2187961  0.487545   
 3  6   6614486   3802821                 2811665  0.574923   
 4  7   9060200   5815220                 3244980  0.641842   
 5  8  11441495   7960139                 3481356  0.695725   
 
    avg_exact_types_per_H  max_exact_types_per_H   V_count   H_count  \
 0               4.293406                    464  50635145  50635145   
 1               2.668892                    599  49955522  49955522   
 2               2.051091                    690  49

## Example Queries

These are sanity checks for the most frequent harmonic classes and the exact n-grams that map into one of those classes.

The `representative_example_ngram` column is only a readable example chosen from the class. The stable class identifier is `harmonic_id`, and the reconstructable matrix representation is `harmonic_key`.

In [8]:
N = 3

con = duckdb.connect(str(DB_PATH), read_only=True)
try:
    ds.configure_connection(con)
    top_harmonic = con.execute(
        """
        SELECT
            harmonic_id,
            count,
            frequency,
            n_exact_ngrams,
            example_ngram AS representative_example_ngram,
            harmonic_key
        FROM harmonic_ngrams
        WHERE n = ?
        ORDER BY count DESC, harmonic_id
        LIMIT 10
        """,
        [N],
    ).fetchdf()

    top_exact_in_top_harmonic = con.execute(
        """
        WITH top_h AS (
            SELECT harmonic_id
            FROM harmonic_ngrams
            WHERE n = ?
            ORDER BY count DESC, harmonic_id
            LIMIT 1
        )
        SELECT
            v.exact_ngram_id,
            v.ngram,
            v.count,
            v.frequency,
            v.harmonic_id
        FROM exact_ngrams v
        JOIN top_h USING (harmonic_id)
        ORDER BY v.count DESC, v.exact_ngram_id
        LIMIT 20
        """,
        [N],
    ).fetchdf()
finally:
    con.close()

top_harmonic, top_exact_in_top_harmonic

(           harmonic_id    count  frequency  n_exact_ngrams  \
 0  H3_ede3c4f53675bbb0  2203877   0.043525             464   
 1  H3_35d6bfda85b78e3b  1907266   0.037667             430   
 2  H3_4e99eecf61b0edec  1827464   0.036091             396   
 3  H3_5f9ccd30c6b35272  1273259   0.025146             357   
 4  H3_614d8a0823c338e1  1183031   0.023364             353   
 5  H3_aef0281c29774fb7  1138503   0.022484             281   
 6  H3_0859351c7b2e5686  1031824   0.020378             355   
 7  H3_cb0f464e362cf3bc   962685   0.019012             239   
 8  H3_2bd39a67fe3535a8   924372   0.018256             334   
 9  H3_950fb5de7bd72a8d   878822   0.017356             220   
 
   representative_example_ngram                            harmonic_key  
 0                        G C G  3:111000000000101010000101000010000000  
 1                        C G C  3:101000010000101000000111000000000010  
 2                        C G D  3:100000011000100000001110000001000010  
 3       

## Deliverable

The deliverable is a validated DuckDB database at `data/processed/harmonic_trends.duckdb`.

Notebook 2 writes this database directly. Any CSV exports should be treated as optional compatibility/reporting outputs, not pipeline inputs.

## Handoff

After this notebook runs, `data/processed/harmonic_trends.duckdb` is the source of truth for global `V_n`/`H_n` counts and exact-to-harmonic mappings.

Notebook 4 uses this database for lightweight target trend tracking. Notebook 7 uses it to build broader support-thresholded harmonic document-term counts.